DNC 판매 모니터링, 재고일수 품절 관리 

In [1]:
import pandas as pd 
import numpy as np 
import glob
import os
from sap_download import download_stockout_prediction, download_inventory_overview
from datetime import datetime
from inventory_utils2 import filter_special_stock
from datetime import date

In [2]:

overview_path = "psi_input\재고개요\재고개요_모니터링.xlsx"
check = pd.read_excel(overview_path)
check.rename(columns = {"자재" : "자재코드", "특별 재고":"특별재고"}, inplace = True)
check_df = filter_special_stock(check)
check_df.to_excel("checkcheck.xlsx")


In [3]:

def _build_po_dataframe():
    stockout_path = "psi_input\품절예상조회\품절예상조회_모니터링.xlsx"
    overview_path = "psi_input\재고개요\재고개요_모니터링.xlsx"
    sf_path       = "psi_input\SF\SF_2603.xlsx"

    df_stockout = pd.read_excel(stockout_path)
    df_overview = pd.read_excel(overview_path)
    df_sf = pd.read_excel(sf_path)
    df_sf.columns = df_sf.columns.astype(str)

    # 품절예상조회 전처리 
    df_stockout = df_stockout[["자재", "자재명", "3개월 평균출하", "당월출하"]].copy()
    df_stockout.rename(columns = {"자재":"자재코드", "자재명":"자재내역", "3개월 평균출하":"3평판"}, inplace=True)
    df_stockout["자재코드"] = df_stockout["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    df_stockout["당월출하"] = pd.to_numeric(df_stockout["당월출하"], errors="coerce").fillna(0)
    df_stockout["3평판"] = pd.to_numeric(df_stockout["3평판"], errors="coerce").fillna(0)


    # SF 전처리 
    today = datetime.today() 
    year_month = today.strftime('%Y.%m')
    df_sf = df_sf[["자재코드", "자재내역", "관리 채널", year_month]].copy()
    df_sf.rename(columns = {"관리 채널":"관리채널", year_month:"SF"}, inplace=True)
    df_sf["자재코드"] = df_sf["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    df_sf["SF"] = pd.to_numeric(df_sf["SF"], errors="coerce").fillna(0)
    df_sf = df_sf.groupby("자재코드").agg(자재내역=("자재내역", "first"), SF=("SF", "sum")).reset_index()
    #df_sf["SF"] = df_sf["SF"].round().astype(int)

    # 품절예상조회 + SF 병합
    # df_standard의 자재내역 기본, 비어있는 경우만 df_sf 자재내역으로 채움
    df_standard = pd.merge(df_stockout, df_sf, on="자재코드", how="outer", suffixes=("", "_sf"))
    df_standard["자재내역"] = df_standard["자재내역"].fillna(df_standard["자재내역_sf"])
    df_standard.drop(columns=["자재내역_sf"], inplace=True)

    df_standard["3평판"]   = df_standard["3평판"].fillna(0)
    df_standard["당월출하"] = df_standard["당월출하"].fillna(0)
    df_standard["SF"]     = df_standard["SF"].fillna(0)

    # # 판매율 계산 (분모 0이면 NaN 처리)
    # df_standard["판매율(평판)"] = (df_standard["당월출하"] / df_standard["3평판"]).where(df_standard["3평판"] != 0)
    # df_standard["판매율(SF)"]  = (df_standard["당월출하"] / df_standard["SF"]).where(df_standard["SF"] != 0)

    df_standard["3평판"] = df_standard["3평판"].astype(float)
    df_standard["당월출하"] = df_standard["당월출하"].astype(float)
    df_standard["SF"] = df_standard["SF"].astype(float)

    # 판매율(SF) 높은 순 정렬
    #df_standard = df_standard.sort_values("판매율(SF)", ascending=False, na_position="last").reset_index(drop=True)
    df_standard = df_standard[["자재코드", "자재내역", "3평판", "SF", "당월출하"]]

    #재고개요 전처리 
    df_overview = df_overview[["자재","자재 내역", "저장 위치", "배치", "특별 재고", "기말 재고 수량"]].copy()
    df_overview.rename(columns = {"자재":"자재코드", "자재 내역":"자재내역", "저장 위치":"저장위치", "배치":"배치", "특별 재고":"특별재고", "기말 재고 수량":"기말재고"}, inplace=True)
    df_overview["자재코드"] = df_overview["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    df_overview["기말재고"] = pd.to_numeric(df_overview["기말재고"], errors="coerce").fillna(0)

    # 특별재고 제거
    df_overview = filter_special_stock(df_overview)

    # 자재코드- 저장위치로 grouping 
    df_overview["자재코드"] = df_overview["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    df_overview["저장위치"] = df_overview["저장위치"].fillna("알수없음")
    df_overview["저장위치"] = df_overview["저장위치"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True)

    df_overview = df_overview.groupby(["자재코드", "저장위치"], as_index = False).agg({
    "자재내역" : "first",
    "기말재고" : "sum"
    })

    # 각 자재코드별 저장위치의 기말재고 피벗 테이블
    df_overview = df_overview.pivot_table(
    index=["자재코드", "자재내역"],
    columns="저장위치",
    values="기말재고",
    aggfunc="sum",   # 중복 시 합계
    fill_value=0     # 없는 값은 0
    )
    # 피벗 후 reset_index로 자재코드, 자재내역 컬럼으로 내리기
    df_overview = df_overview.reset_index()
    df_overview.columns.name = None

    # merge (자재코드 기준, 자재내역은 suffix로 분리)
    df_standard = pd.merge(df_standard, df_overview, on="자재코드", how="outer", suffixes=("", "_ov"))

    # df_standard 자재내역 비어있으면 df_overview 자재내역으로 채우기
    df_standard["자재내역"] = df_standard["자재내역"].fillna(df_standard["자재내역_ov"])
    df_standard.drop(columns=["자재내역_ov"], inplace=True)


    return df_standard

<>:4: SyntaxWarning: invalid escape sequence '\S'
<>:4: SyntaxWarning: invalid escape sequence '\S'
C:\Users\USER\AppData\Local\Temp\ipykernel_19652\1626964692.py:4: SyntaxWarning: invalid escape sequence '\S'
  sf_path       = "psi_input\SF\SF_2603.xlsx"


In [4]:
standard_df = _build_po_dataframe()

In [5]:
display(standard_df.head())

,자재코드,자재내역,3평판,SF,당월출하,5000,5010,5100,5400,5600,...,7000,7020,7030,7040,7050,7060,7070,7080,7090,알수없음
0,1000940,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,NaN,NaN,NaN,117800.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,449501.195
1,1300271,[원료]대웅제약_DW-EGF동결원액_4mg/1ml,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1201.960
2,2301915,(단종)[판촉물]쇼핑백_23년프로모션(245*95*230),NaN,NaN,NaN,13.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000
3,2302226,[세트포장재_면세슬리브]멜라토닝패치4매*3박스,NaN,NaN,NaN,2324.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000
4,2302396,(단종)[세트포장재_슬리브]멜라비타씨토너_한가인싸인(판촉몰),NaN,NaN,NaN,13500.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000


In [6]:
standard_df.to_excel("after 1.xlsx")

In [7]:
df = standard_df

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 28 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   자재코드    1202 non-null   object 
 1   자재내역    1202 non-null   object 
 2   3평판     1121 non-null   float64
 3   SF      1121 non-null   float64
 4   당월출하    1121 non-null   float64
 5   5000    612 non-null    float64
 6   5010    612 non-null    float64
 7   5100    612 non-null    float64
 8   5400    612 non-null    float64
 9   5600    612 non-null    float64
 10  6010    612 non-null    float64
 11  6020    612 non-null    float64
 12  6030    612 non-null    float64
 13  6050    612 non-null    float64
 14  6060    612 non-null    float64
 15  6080    612 non-null    float64
 16  6090    612 non-null    float64
 17  6092    612 non-null    float64
 18  7000    612 non-null    float64
 19  7020    612 non-null    float64
 20  7030    612 non-null    float64
 21  7040    612 non-null    float64
 22  

In [9]:
df.columns

Index(['자재코드', '자재내역', '3평판', 'SF', '당월출하', '5000', '5010', '5100', '5400',
       '5600', '6010', '6020', '6030', '6050', '6060', '6080', '6090', '6092',
       '7000', '7020', '7030', '7040', '7050', '7060', '7070', '7080', '7090',
       '알수없음'],
      dtype='object')

In [10]:
warehouse_cols = ['5000', '5010', '5100', '5400', '5600', '6010', '6020', '6030',
                  '6040', '6050', '6060', '6080', '6090', '7000', '7020', '7030',
                  '7040', '7050', '7060', '7070', '7080', '7090', '알수없음', '6092']

# warehouse_cols = ['5000', '5010', '5100', '5400', '5600', '6010', '6020', '6030', '6050', '6060', '6080', '6090', '7000', '7020', '7030',
#                   '7040', '7050', '7060', '7070', '7080', '7090', '알수없음']

# df[warehouse_cols] = df[warehouse_cols].fillna(0)
# df["총재고"] = df[warehouse_cols].sum(axis=1)
valid_warehouse_cols = [col for col in warehouse_cols if col in df.columns]
df[valid_warehouse_cols] = df[valid_warehouse_cols].fillna(0)
df["총재고"] = df[valid_warehouse_cols].sum(axis=1)

In [11]:
df["이지+천지재고"] = df["5000"] + df["6090"]

In [12]:
df.to_excel("after 2.xlsx")

In [13]:
order_path= "psi_input\오더집계\오더집계.xlsx"

order_df = pd.read_excel(order_path)

In [14]:
display(order_df.head())

,년월,유통경로,유통경로명,담당자,담당자명,본부,본부명,사업부,사업부명,사무소,...,통화(외화),매출금액(외화),반품금액(외화),사전할인(외화),사후할인발생(외화),사후할인집행(외화),매출할인(외화),환율,단위,오더(청구)유형
0,2026.04,10,내수,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0,0,0,0,0,1.0,EA,ZFD
1,2026.04,10,내수,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0,0,0,0,0,1.0,EA,ZFD
2,2026.04,10,내수,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0,0,0,0,0,1.0,EA,ZFD
3,2026.04,10,내수,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0,0,0,0,0,1.0,EA,ZFD
4,2026.04,10,내수,1.202120e+09,주철승,1202.0,DW_ETC로컬본부,1203.0,DW_서울3사업부,A22,...,NaN,0.0,0,0,0,0,0,1.0,EA,ZON


In [15]:
order_df.rename(columns = {"자재" : "자재코드", "자재명" : "자재내역"}, inplace = True)

In [16]:
order_df["자재코드"] = (order_df["자재코드"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True))
order_df["순매출수량"] = pd.to_numeric(order_df["순매출수량"], errors="coerce")

In [17]:
order_df = order_df.groupby("자재코드", as_index=False).agg({
    "자재내역": "first",
    "순매출수량": "sum"
})

In [18]:
display(order_df)

,자재코드,자재내역,순매출수량
0,2302428,(단종)[세트포장재]이지듀2024가정의달vIP싸바리_쇼핑백,0
1,2303084,(단종)[판촉물_쇼핑백]미스트로즈쇼핑백,10
2,2303248,(단종)[세트포장재]원데이앰플_로즈에디션쇼핑백(박스형),0
3,7001012,(단종)[판촉물_상품]클렌징브러쉬듀오,11
4,7300456,=배송비=,13792
...,...,...,...
309,9401227,[카카오] 봄에디션+리필+포스트레이저크림 5ml 23호,48
310,9401231,[판촉몰] 골든엘릭서 오일 4ML 3개,689
311,9401327,이지듀EGFx다운타임마스크20ml(미품),0
312,9401548,[세트]앰플쿠션19호(15g)+글로우립로즈(카카오),10


In [19]:
order_df.to_excel("after 3.xlsx")

In [20]:
standard_df = standard_df.merge(
    order_df[["자재코드","자재내역", "순매출수량"]],
    on="자재코드",
    how="outer"
)

In [21]:
not_warehouse_cols = ['5010', '5100', '5400', '5600', '6010', '6020', '6030',
                  '6040', '6050', '6060', '6080', '7000', '7020', '7030',
                  '7040', '7050', '7060', '7070', '7080', '7090', '알수없음', '6092']


standard_df = standard_df.drop(columns=not_warehouse_cols, errors='ignore')

In [22]:
standard_df["순매출수량"] = standard_df["순매출수량"].fillna(0)

In [23]:
display(standard_df.head())

,자재코드,자재내역_x,3평판,SF,당월출하,5000,6090,총재고,이지+천지재고,자재내역_y,순매출수량
0,1000940,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,NaN,NaN,NaN,117800.0,0.0,567301.195,117800.0,NaN,0.0
1,1300271,[원료]대웅제약_DW-EGF동결원액_4mg/1ml,NaN,NaN,NaN,0.0,0.0,1201.960,0.0,NaN,0.0
2,2301915,(단종)[판촉물]쇼핑백_23년프로모션(245*95*230),NaN,NaN,NaN,13.0,0.0,13.000,13.0,NaN,0.0
3,2302226,[세트포장재_면세슬리브]멜라토닝패치4매*3박스,NaN,NaN,NaN,2324.0,0.0,2324.000,2324.0,NaN,0.0
4,2302396,(단종)[세트포장재_슬리브]멜라비타씨토너_한가인싸인(판촉몰),NaN,NaN,NaN,13500.0,0.0,13500.000,13500.0,NaN,0.0


In [24]:
standard_df["자재내역_x"] =standard_df["자재내역_x"].fillna(standard_df["자재내역_y"])
standard_df = standard_df.drop(columns = "자재내역_y")
standard_df.rename(columns = {"자재내역_x" : "자재내역"}, inplace = True) 

In [25]:
standard_df.columns

Index(['자재코드', '자재내역', '3평판', 'SF', '당월출하', '5000', '6090', '총재고', '이지+천지재고',
       '순매출수량'],
      dtype='object')

In [26]:
standard_df["3평판"] = pd.to_numeric(standard_df["3평판"], errors="coerce").fillna(0)
standard_df["SF"] = pd.to_numeric(standard_df["SF"], errors="coerce").fillna(0)
standard_df["순매출수량"] = pd.to_numeric(standard_df["순매출수량"], errors="coerce").fillna(0)

    # 판매율 계산 (분모 0이면 NaN 처리)
standard_df["판매율(평판)"] = (standard_df["순매출수량"] / standard_df["3평판"]).where(standard_df["3평판"] != 0)
standard_df["판매율(SF)"]  = (standard_df["순매출수량"] / standard_df["SF"]).where(standard_df["SF"] != 0)

In [27]:
display(standard_df.head())

,자재코드,자재내역,3평판,SF,당월출하,5000,6090,총재고,이지+천지재고,순매출수량,판매율(평판),판매율(SF)
0,1000940,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,0.0,0.0,NaN,117800.0,0.0,567301.195,117800.0,0.0,NaN,NaN
1,1300271,[원료]대웅제약_DW-EGF동결원액_4mg/1ml,0.0,0.0,NaN,0.0,0.0,1201.960,0.0,0.0,NaN,NaN
2,2301915,(단종)[판촉물]쇼핑백_23년프로모션(245*95*230),0.0,0.0,NaN,13.0,0.0,13.000,13.0,0.0,NaN,NaN
3,2302226,[세트포장재_면세슬리브]멜라토닝패치4매*3박스,0.0,0.0,NaN,2324.0,0.0,2324.000,2324.0,0.0,NaN,NaN
4,2302396,(단종)[세트포장재_슬리브]멜라비타씨토너_한가인싸인(판촉몰),0.0,0.0,NaN,13500.0,0.0,13500.000,13500.0,0.0,NaN,NaN


In [28]:
numeric_cols = ['3평판', 'SF', '당월출하', '판매율(평판)', '판매율(SF)', '5000',
                '6090', '총재고', '이지+천지재고', '순매출수량']

standard_df[numeric_cols] = standard_df[numeric_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

In [29]:
standard_df.to_excel("after 4.xlsx")

In [30]:
display(standard_df.head())

,자재코드,자재내역,3평판,SF,당월출하,5000,6090,총재고,이지+천지재고,순매출수량,판매율(평판),판매율(SF)
0,1000940,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,0.0,0.0,0.0,117800.0,0.0,567301.195,117800.0,0.0,0.0,0.0
1,1300271,[원료]대웅제약_DW-EGF동결원액_4mg/1ml,0.0,0.0,0.0,0.0,0.0,1201.960,0.0,0.0,0.0,0.0
2,2301915,(단종)[판촉물]쇼핑백_23년프로모션(245*95*230),0.0,0.0,0.0,13.0,0.0,13.000,13.0,0.0,0.0,0.0
3,2302226,[세트포장재_면세슬리브]멜라토닝패치4매*3박스,0.0,0.0,0.0,2324.0,0.0,2324.000,2324.0,0.0,0.0,0.0
4,2302396,(단종)[세트포장재_슬리브]멜라비타씨토너_한가인싸인(판촉몰),0.0,0.0,0.0,13500.0,0.0,13500.000,13500.0,0.0,0.0,0.0


In [31]:
standard_df["재고대응률"] = standard_df.apply(
    lambda row: row["이지+천지재고"] / row["SF"] if row["SF"] > 0 else "계획없음", axis=1
)

In [32]:
display(standard_df.head())

,자재코드,자재내역,3평판,SF,당월출하,5000,6090,총재고,이지+천지재고,순매출수량,판매율(평판),판매율(SF),재고대응률
0,1000940,[원료]DW-EGFLiposol(EGF-나노리포좀)250ppm 1g,0.0,0.0,0.0,117800.0,0.0,567301.195,117800.0,0.0,0.0,0.0,계획없음
1,1300271,[원료]대웅제약_DW-EGF동결원액_4mg/1ml,0.0,0.0,0.0,0.0,0.0,1201.960,0.0,0.0,0.0,0.0,계획없음
2,2301915,(단종)[판촉물]쇼핑백_23년프로모션(245*95*230),0.0,0.0,0.0,13.0,0.0,13.000,13.0,0.0,0.0,0.0,계획없음
3,2302226,[세트포장재_면세슬리브]멜라토닝패치4매*3박스,0.0,0.0,0.0,2324.0,0.0,2324.000,2324.0,0.0,0.0,0.0,계획없음
4,2302396,(단종)[세트포장재_슬리브]멜라비타씨토너_한가인싸인(판촉몰),0.0,0.0,0.0,13500.0,0.0,13500.000,13500.0,0.0,0.0,0.0,계획없음


In [33]:
standard_df.to_excel("after 5.xlsx")

In [34]:
from datetime import date

In [35]:
today = date.today()
first_of_month = today.replace(day=1)
elapsed = (today - first_of_month).days

print(elapsed)  # 오늘이 7일이면 6 출력

13


In [36]:
standard_df["일평균소진"] = standard_df["순매출수량"] / elapsed

In [37]:
standard_df.to_excel("after 6.xlsx")

In [38]:
standard_df["이지+천지재고일수"] = standard_df.apply(
    lambda row: row["이지+천지재고"] / row["일평균소진"] if row["일평균소진"] > 0 else "일평균소진없음", axis=1
)

In [39]:
# standard_df["이지+천지재고일수(3평판)"] = standard_df.apply(
#     lambda row: (row["이지+천지재고"] / row["3평판"])*30 if row["3평판"] > 0 else "평판없음", axis=1
# )

In [40]:
standard_df["이지 재고일수"] = standard_df.apply(
    lambda row: row["5000"] / row["일평균소진"] if row["일평균소진"] > 0 else "일평균소진없음", axis = 1
)

In [41]:
standard_df["총재고일수"] = standard_df.apply(
    lambda row: row["총재고"] / row["일평균소진"] if row["일평균소진"] > 0 else "일평균소진없음", axis=1
)

In [42]:
standard_df.to_excel("after 7.xlsx")

In [43]:
standard_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1202 entries, 0 to 1201
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   자재코드       1202 non-null   object 
 1   자재내역       1202 non-null   object 
 2   3평판        1202 non-null   float64
 3   SF         1202 non-null   float64
 4   당월출하       1202 non-null   float64
 5   5000       1202 non-null   float64
 6   6090       1202 non-null   float64
 7   총재고        1202 non-null   float64
 8   이지+천지재고    1202 non-null   float64
 9   순매출수량      1202 non-null   float64
 10  판매율(평판)    1202 non-null   float64
 11  판매율(SF)    1202 non-null   float64
 12  재고대응률      1202 non-null   object 
 13  일평균소진      1202 non-null   float64
 14  이지+천지재고일수  1202 non-null   object 
 15  이지 재고일수    1202 non-null   object 
 16  총재고일수      1202 non-null   object 
dtypes: float64(11), object(6)
memory usage: 159.8+ KB


In [44]:
# 1. 먼저 숫자형으로 안전하게 변환
standard_df["이지+천지재고일수"] = pd.to_numeric(standard_df["이지+천지재고일수"], errors='coerce')
# 2. 292년 (약 106,580일) 기준 설정
LIMIT_DAYS_292Y = 106580 
def handle_long_term_stock(days):
    if pd.isna(days) or days < 0:
        return " "
    
    # 292년을 초과하는 경우
    if days > LIMIT_DAYS_292Y:
        return "292년 이상"
    
    try:
        # 정상 범위는 날짜 계산
        return (today + pd.Timedelta(days=days)).strftime("%Y-%m-%d")
    except OverflowError:
        # 계산 중 혹시 모를 오버플로우 발생 시
        return "292년 이상"
# 적용
standard_df["품절예상일"] = standard_df["이지+천지재고일수"].apply(handle_long_term_stock)

In [45]:
standard_df.to_excel("after 8.xlsx")

In [46]:
def classify_risk(days):
    
    if pd.isna(days) or days < 0:
        return "데이터 없음"
    
    if days < 45:
        return "위험"
    elif days < 60:
        return "경고"
    elif days < 90:
        return "안전"
    else:
        return "과재고"

In [47]:
# 새로운 위험도 컬럼 생성
standard_df["위험도"] = standard_df["이지+천지재고일수"].apply(classify_risk)

In [48]:
# standard_df["이지+천지재고일수(3평판)"] = pd.to_numeric(standard_df["이지+천지재고일수(3평판)"], errors='coerce')

In [49]:
# # 새로운 위험도 컬럼 생성
# standard_df["위험도(3평판)"] = standard_df["이지+천지재고일수(3평판)"].apply(classify_risk)


In [50]:
# def dong_classify_risk(days):
#     # 0이거나(판매없음 시 0으로 계산되는 경우) 데이터가 없는 경우
#     if pd.isna(days) or days <= 0:
#         return "판매없음"
    
#     # 40일 미만인 경우
#     if days < 40:
#         return "주의"
    
#     # 그 외 모든 경우 (40일 이상)
#     else:
#         return "안전"

# standard_df["위험도(엑셀관리)"] = standard_df["이지+천지재고일수(일평균소진)"].apply(dong_classify_risk)

In [51]:
standard_df["판매율(평판)"] = standard_df["판매율(평판)"] * 100
standard_df["판매율(SF)"] = standard_df["판매율(SF)"] * 100

In [52]:
# 재고대응률 계산 후, "계획없음"이 아닌 값만 * 100
standard_df["재고대응률"] = standard_df["재고대응률"].where(
    standard_df["재고대응률"] == "계획없음",
    other=standard_df["재고대응률"] * 100
)

In [53]:
standard_df = standard_df[["자재코드", "자재내역", "3평판", "SF", "당월출하", "판매율(평판)", "판매율(SF)", "5000", "6090", "총재고", 
"이지+천지재고", "순매출수량", "재고대응률", "일평균소진", "이지+천지재고일수", "이지 재고일수", "총재고일수", "품절예상일", "위험도"]]

In [54]:
standard_df = standard_df.rename(columns = {"5000" : "이지 재고", "6090" : "천지 재고"})

In [55]:
standard_df.to_excel("전체 위험도 관리.xlsx")

In [56]:
danger_df = standard_df[standard_df["위험도"] == "위험"].reset_index(drop=True)

In [57]:
is_단종 = danger_df["자재내역"].str.contains("단종", na=False, regex=False)
df_normal = danger_df[~is_단종].sort_values("이지+천지재고일수", ascending=True)
df_단종   = danger_df[is_단종].sort_values("이지+천지재고일수", ascending=True)
danger_df = pd.concat([df_normal, df_단종]).reset_index(drop=True)

In [58]:
danger_df.to_excel("미출 위험 모니터링.xlsx")

In [59]:
today = date.today().strftime("%Y%m%d")
filename = f"{today}_모니터링.xlsx"
with pd.ExcelWriter(filename) as writer:
    danger_df.to_excel(writer, sheet_name="위험 관리 필요", index=False)
    standard_df.to_excel(writer, sheet_name="전체 자재 확인용(참고)", index=False)